In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NA_7,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,label
0,1,0,0,0,0,0,0,1,1.0,0.0,...,0.001816,0.001211,0.000560,0.000000,0.000000,0.000161,0.0,0.000301,0.000818,1.0
1,1,1,1,1,0,0,0,0,1.0,1.0,...,0.005029,0.000784,0.000802,0.001592,0.001612,0.000741,0.0,0.000545,0.002437,1.0
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


In [3]:
### Dataset TH
top_features = ['ND', 'BL', 'AS_NAC', 'NAC', 'AI']

selected_columns = [
    col for col in TH.columns 
    if any(col.startswith(f + '_') for f in top_features)
]


Selected_TH = TH[selected_columns]
Selected_TH['label']=TH['label']

print("Number of features:", len(selected_columns))
Selected_TH.columns

### Dataset TW
top_features = ['NAC', 'NAD', 'NCD', 'AS(NAC)', 'NA']

selected_columns = [
    col for col in TW.columns 
    if any(col.startswith(f + '_') for f in top_features)
]


Selected_TW = TW[selected_columns]
Selected_TW['label']=TW['label']

print("Number of features:", len(selected_columns))
Selected_TW.columns


Number of features: 40
Number of features: 35


Index(['NCD_0', 'NCD_1', 'NCD_2', 'NCD_3', 'NCD_4', 'NCD_5', 'NCD_6', 'NAC_0',
       'NAC_1', 'NAC_2', 'NAC_3', 'NAC_4', 'NAC_5', 'NAC_6', 'AS(NAC)_0',
       'AS(NAC)_1', 'AS(NAC)_2', 'AS(NAC)_3', 'AS(NAC)_4', 'AS(NAC)_5',
       'AS(NAC)_6', 'NA_0', 'NA_1', 'NA_2', 'NA_3', 'NA_4', 'NA_5', 'NA_6',
       'NAD_0', 'NAD_1', 'NAD_2', 'NAD_3', 'NAD_4', 'NAD_5', 'NAD_6', 'label'],
      dtype='object')

### Twiter

In [4]:
X = Selected_TW.drop(columns=["label"])
y = Selected_TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [5]:
lr_best = LogisticRegression(max_iter=2000,random_state=42)
lr_best.fit(X_train, y_train)

pred_lr = lr_best.predict(X_test)
pred_proba = lr_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best Logistic Regression ")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best Logistic Regression 
Accuracy: 0.9676284556890058
Precision: 0.9414448669201521
Recall: 0.8914491449144915
F1-score: 0.9157651410078594
ROC-AUC: 0.9925870472556619


In [6]:
rf_best = RandomForestClassifier( n_estimators=200, max_depth=None, max_features=0.5, min_samples_leaf=1, random_state=42)
rf_best.fit(X_train, y_train)

pred_rf = rf_best.predict(X_test)
pred_proba = rf_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best Random Forest ")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best Random Forest 
Accuracy: 0.9687655461587663
Precision: 0.9307295504789977
Recall: 0.9094509450945094
F1-score: 0.9199672220704725
ROC-AUC: 0.9918370160830665


In [7]:
xgb_best = XGBClassifier( n_estimators=100,max_depth=5,learning_rate=0.1,subsample=0.8,random_state=42,eval_metric='logloss', use_label_encoder=False)
xgb_best.fit(X_train, y_train)

pred_xgb = xgb_best.predict(X_test)
pred_proba = xgb_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_xgb)
precision = precision_score(y_test, pred_xgb, zero_division=0)
recall = recall_score(y_test, pred_xgb, zero_division=0)
f1 = f1_score(y_test, pred_xgb, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best XGBoost")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best XGBoost
Accuracy: 0.9689787506218464
Precision: 0.9351301115241636
Recall: 0.9056705670567057
F1-score: 0.920164609053498
ROC-AUC: 0.993120765124726


In [9]:
from sklearn.ensemble import StackingClassifier
estimators = [
    ('lr', lr_best),
    ('rf', rf_best),
    ('xgb', xgb_best)
]

stacking_model = StackingClassifier(estimators=estimators,final_estimator=LogisticRegression(),cv=5,n_jobs=-1)

stacking_model.fit(X_train, y_train)

pred_stack = stacking_model.predict(X_test)
pred_proba = stacking_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_stack)
precision = precision_score(y_test, pred_stack, zero_division=0)
recall = recall_score(y_test, pred_stack, zero_division=0)
f1 = f1_score(y_test, pred_stack, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("=== Stacking Ensemble (LR + RF + XGB) ===")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

=== Stacking Ensemble (LR + RF + XGB) ===
Accuracy: 0.9693696254708265
Precision: 0.9385161652027658
Recall: 0.904050405040504
F1-score: 0.9209609389326976
ROC-AUC: 0.9931108106161924


In [15]:
import pandas as pd

data = {
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Stacking Ensemble"
    ],
    "Accuracy": [0.9676, 0.9688, 0.9690, 0.9694],
    "Precision": [0.9414, 0.9307, 0.9351, 0.9385],
    "Recall": [0.8914, 0.9095, 0.9057, 0.9041],
    "F1-score": [0.9158, 0.9200, 0.9202, 0.9210],
    "ROC-AUC": [0.9926, 0.9918, 0.9931, 0.9931]
}

df_final_tw = pd.DataFrame(data)
df_final_tw

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.9676,0.9414,0.8914,0.9158,0.9926
1,Random Forest,0.9688,0.9307,0.9095,0.9200,0.9918
2,XGBoost,0.9690,0.9351,0.9057,0.9202,0.9931
3,Stacking Ensemble,0.9694,0.9385,0.9041,0.9210,0.9931


### TH

In [10]:
X = Selected_TH.drop(columns=["label"])
y = Selected_TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [11]:
lr_best = LogisticRegression(max_iter=2000,random_state=42)
lr_best.fit(X_train, y_train)

pred_lr = lr_best.predict(X_test)
pred_proba = lr_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best Logistic Regression ")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best Logistic Regression 
Accuracy: 0.963314358001265
Precision: 0.9730848861283644
Recall: 0.9670781893004116
F1-score: 0.9700722394220846
ROC-AUC: 0.9945451289640307


In [12]:
rf_best = RandomForestClassifier( n_estimators=200, max_depth=None, max_features=0.5, min_samples_leaf=1, random_state=42)
rf_best.fit(X_train, y_train)

pred_rf = rf_best.predict(X_test)
pred_proba = rf_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best Random Forest ")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best Random Forest 
Accuracy: 0.9683744465528147
Precision: 0.9782157676348547
Recall: 0.970164609053498
F1-score: 0.9741735537190083
ROC-AUC: 0.9957031698730294


In [13]:
xgb_best = XGBClassifier( n_estimators=100,max_depth=5,learning_rate=0.1,subsample=0.8,random_state=42,eval_metric='logloss', use_label_encoder=False)
xgb_best.fit(X_train, y_train)

pred_xgb = xgb_best.predict(X_test)
pred_proba = xgb_best.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_xgb)
precision = precision_score(y_test, pred_xgb, zero_division=0)
recall = recall_score(y_test, pred_xgb, zero_division=0)
f1 = f1_score(y_test, pred_xgb, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Best XGBoost")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Best XGBoost
Accuracy: 0.969639468690702
Precision: 0.9782608695652174
Recall: 0.9722222222222222
F1-score: 0.9752321981424149
ROC-AUC: 0.9963679242095589


In [14]:
from sklearn.ensemble import StackingClassifier
estimators = [
    ('lr', lr_best),
    ('rf', rf_best),
    ('xgb', xgb_best)
]

stacking_model = StackingClassifier(estimators=estimators,final_estimator=LogisticRegression(),cv=5,n_jobs=-1)

stacking_model.fit(X_train, y_train)

pred_stack = stacking_model.predict(X_test)
pred_proba = stacking_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_stack)
precision = precision_score(y_test, pred_stack, zero_division=0)
recall = recall_score(y_test, pred_stack, zero_division=0)
f1 = f1_score(y_test, pred_stack, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("=== Stacking Ensemble (LR + RF + XGB) ===")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

=== Stacking Ensemble (LR + RF + XGB) ===
Accuracy: 0.9690069576217584
Precision: 0.9792315680166147
Recall: 0.970164609053498
F1-score: 0.9746770025839794
ROC-AUC: 0.9962344665409799


In [16]:
import pandas as pd

data = {
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Stacking Ensemble"
    ],
    "Accuracy": [0.9633, 0.9684, 0.9696, 0.9690],
    "Precision": [0.9731, 0.9782, 0.9783, 0.9792],
    "Recall": [0.9671, 0.9702, 0.9722, 0.9702],
    "F1-score": [0.9701, 0.9742, 0.9752, 0.9747],
    "ROC-AUC": [0.9945, 0.9957, 0.9964, 0.9962]
}

df_final_th = pd.DataFrame(data)
df_final_th

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.9633,0.9731,0.9671,0.9701,0.9945
1,Random Forest,0.9684,0.9782,0.9702,0.9742,0.9957
2,XGBoost,0.9696,0.9783,0.9722,0.9752,0.9964
3,Stacking Ensemble,0.9690,0.9792,0.9702,0.9747,0.9962
